In [27]:
import pandas as pd
import numpy as np

In [83]:
df_qual = pd.read_parquet(r"C:\Users\Asus\Desktop\Formula1\data\bronze\qualifying\all_seasons_qualifying.parquet")
df_qual.head()

,season,round_number,race_name,circuit_ref,race_date,driver_ref,driver_code,driver_number,constructor_ref,quali_position,q1_time,q2_time,q3_time
0,2018,1,Australian Grand Prix,albert_park,2018-03-25,hamilton,HAM,44,mercedes,1,1:22.824,1:22.051,1:21.164
1,2018,1,Australian Grand Prix,albert_park,2018-03-25,raikkonen,RAI,7,ferrari,2,1:23.096,1:22.507,1:21.828
2,2018,1,Australian Grand Prix,albert_park,2018-03-25,vettel,VET,5,ferrari,3,1:23.348,1:21.944,1:21.838
3,2018,1,Australian Grand Prix,albert_park,2018-03-25,max_verstappen,VER,33,red_bull,4,1:23.483,1:22.416,1:21.879
4,2018,1,Australian Grand Prix,albert_park,2018-03-25,ricciardo,RIC,3,red_bull,5,1:23.494,1:22.897,1:22.152


- no **PRIMARY KEY**
- combination of season and round_number is unique - **PRIMARY KEY**

In [84]:
df_qual.shape

(3455, 13)

In [85]:
df_qual.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3455 entries, 0 to 3454
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   season           3455 non-null   int64 
 1   round_number     3455 non-null   int64 
 2   race_name        3455 non-null   object
 3   circuit_ref      3455 non-null   object
 4   race_date        3455 non-null   object
 5   driver_ref       3455 non-null   object
 6   driver_code      3455 non-null   object
 7   driver_number    3455 non-null   int64 
 8   constructor_ref  3455 non-null   object
 9   quali_position   3455 non-null   int64 
 10  q1_time          3455 non-null   object
 11  q2_time          2563 non-null   object
 12  q3_time          1702 non-null   object
dtypes: int64(4), object(9)
memory usage: 351.0+ KB


In [86]:
df_qual["qual_ref"] = df_qual["season"].astype(str) + "_" + df_qual["round_number"].astype(str)

In [87]:
df_qual.head()

,season,round_number,race_name,circuit_ref,race_date,driver_ref,driver_code,driver_number,constructor_ref,quali_position,q1_time,q2_time,q3_time,qual_ref
0,2018,1,Australian Grand Prix,albert_park,2018-03-25,hamilton,HAM,44,mercedes,1,1:22.824,1:22.051,1:21.164,2018_1
1,2018,1,Australian Grand Prix,albert_park,2018-03-25,raikkonen,RAI,7,ferrari,2,1:23.096,1:22.507,1:21.828,2018_1
2,2018,1,Australian Grand Prix,albert_park,2018-03-25,vettel,VET,5,ferrari,3,1:23.348,1:21.944,1:21.838,2018_1
3,2018,1,Australian Grand Prix,albert_park,2018-03-25,max_verstappen,VER,33,red_bull,4,1:23.483,1:22.416,1:21.879,2018_1
4,2018,1,Australian Grand Prix,albert_park,2018-03-25,ricciardo,RIC,3,red_bull,5,1:23.494,1:22.897,1:22.152,2018_1


- changing datatype of race_date

In [88]:
df_qual["race_date"] = pd.to_datetime(df_qual["race_date"])

- dropping driver_code and driver_number columns

In [89]:
df_qual.drop(columns=["driver_code", "driver_number"], inplace=True)

- handling q1_time, q2_time and q3_time columns

In [90]:
# problematic values in q1_time, q2_time, q3_time

a = df_qual[
    ~df_qual["q1_time"].astype(str).str.contains(":", na=False)
]["q1_time"].unique()

b = df_qual[
    ~df_qual["q2_time"].astype(str).str.contains(":", na=False)
]["q2_time"].unique()

c = df_qual[
    ~df_qual["q3_time"].astype(str).str.contains(":", na=False)
]["q3_time"].unique()

print("-------------------- q1_time --------------------")
print(a, "\n")

print("-------------------- q2_time --------------------")
print(b, "\n")

print("-------------------- q3_time --------------------")
print(c)

-------------------- q1_time --------------------
['' '53.904' '54.160' '54.037' '54.249' '54.236' '54.346' '54.388'
 '54.450' '54.207' '54.595' '54.309' '54.620' '54.301' '54.523' '54.194'
 '54.705' '54.796' '54.892' '54.963' '55.426'] 

-------------------- q2_time --------------------
[None '53.803' '53.819' '53.647' '53.825' '53.787' '53.856' '53.871'
 '53.818' '53.941' '53.840' '53.995' '54.026' '54.175' '54.377' '54.693'
 ''] 

-------------------- q3_time --------------------
[None '53.377' '53.403' '53.433' '53.613' '53.790' '53.906' '53.957'
 '54.010' '54.154' '54.200' '']


In [91]:
def qual_time_to_seconds(time_str):

    # Missing values
    if pd.isna(time_str) or str(time_str).strip() == "":
        return np.nan

    time_str = str(time_str)

    # Format: 1:22.824
    if ":" in time_str:
        mins, secs = time_str.split(":")
        return int(mins) * 60 + float(secs)

    # Format: 53.904
    return float(time_str)

In [92]:
time_cols = ["q1_time", "q2_time", "q3_time"]

for col in time_cols:
    df_qual[col + "_sec"] = df_qual[col].apply(qual_time_to_seconds)

- The number of values in **"q1_time"** and **"q1_time_sec"** differ because q1_time contains empty strings (''), which Pandas counts as non-null values, whereas the conversion function converts those empty strings to NaN, reducing the non-null count in q1_time_sec.

In [93]:
df_qual.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3455 entries, 0 to 3454
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   season           3455 non-null   int64         
 1   round_number     3455 non-null   int64         
 2   race_name        3455 non-null   object        
 3   circuit_ref      3455 non-null   object        
 4   race_date        3455 non-null   datetime64[ns]
 5   driver_ref       3455 non-null   object        
 6   constructor_ref  3455 non-null   object        
 7   quali_position   3455 non-null   int64         
 8   q1_time          3455 non-null   object        
 9   q2_time          2563 non-null   object        
 10  q3_time          1702 non-null   object        
 11  qual_ref         3455 non-null   object        
 12  q1_time_sec      3412 non-null   float64       
 13  q2_time_sec      2548 non-null   float64       
 14  q3_time_sec      1688 non-null   float64

- suspicious rows where Q2 or Q3 qualifying times are present, but the corresponding Q1 qualifying time is missing

In [94]:
df_qual[
    df_qual["q1_time_sec"].isna() &
    (
        df_qual["q2_time_sec"].notna() |
        df_qual["q3_time_sec"].notna()
    )
]

,season,round_number,race_name,circuit_ref,race_date,driver_ref,constructor_ref,quali_position,q1_time,q2_time,q3_time,qual_ref,q1_time_sec,q2_time_sec,q3_time_sec
2786,2024,15,Dutch Grand Prix,zandvoort,2024-08-25,albon,williams,10,,1:10.768,1:10.653,2024_15,NaN,70.768,70.653
2830,2024,17,Azerbaijan Grand Prix,baku,2024-09-15,gasly,alpine,15,,1:43.179,None,2024_17,NaN,103.179,NaN


- dropping rows with inconsistent qualifying data where Q2/Q3 times exist but the Q1 time is missing

In [95]:
df_qual = df_qual[
    ~(
        df_qual["q1_time_sec"].isna() &
        (
            df_qual["q2_time_sec"].notna() |
            df_qual["q3_time_sec"].notna()
        )
    )
]

- q1_time, q2_time and q3_time are redundant columns now - dropping them

In [96]:
df_qual.drop(columns=["q1_time", "q2_time", "q3_time"], inplace=True)

- deriving new columns

In [97]:
# best qualifying time of the driver in a particular race

df_qual["best_quali_time_sec"] = df_qual["q3_time_sec"].fillna(df_qual["q2_time_sec"]).fillna(df_qual["q1_time_sec"])

In [98]:
# gap to pole

pole_times = (
    df_qual
    .groupby(["season", "round_number"])["best_quali_time_sec"]
    .min()
    .reset_index(name="pole_time_sec")
)


df_qual = df_qual.merge(
    pole_times,
    on=["season", "round_number"],
    how="left"
)

df_qual["gap_to_pole"] = (
    (df_qual["best_quali_time_sec"] - df_qual["pole_time_sec"])
)

df_qual.drop(columns=["pole_time_sec"], inplace=True)

df_qual.head()

,season,round_number,race_name,circuit_ref,race_date,driver_ref,constructor_ref,quali_position,qual_ref,q1_time_sec,q2_time_sec,q3_time_sec,best_quali_time_sec,gap_to_pole
0,2018,1,Australian Grand Prix,albert_park,2018-03-25,hamilton,mercedes,1,2018_1,82.824,82.051,81.164,81.164,0.000
1,2018,1,Australian Grand Prix,albert_park,2018-03-25,raikkonen,ferrari,2,2018_1,83.096,82.507,81.828,81.828,0.664
2,2018,1,Australian Grand Prix,albert_park,2018-03-25,vettel,ferrari,3,2018_1,83.348,81.944,81.838,81.838,0.674
3,2018,1,Australian Grand Prix,albert_park,2018-03-25,max_verstappen,red_bull,4,2018_1,83.483,82.416,81.879,81.879,0.715
4,2018,1,Australian Grand Prix,albert_park,2018-03-25,ricciardo,red_bull,5,2018_1,83.494,82.897,82.152,82.152,0.988


In [101]:
# session progression flags 
df_qual["made_q2"] = df_qual["q2_time_sec"].notna()
df_qual["made_q3"] = df_qual["q3_time_sec"].notna()

In [105]:
df_qual.columns

Index(['season', 'round_number', 'race_name', 'circuit_ref', 'race_date',
       'driver_ref', 'constructor_ref', 'quali_position', 'qual_ref',
       'q1_time_sec', 'q2_time_sec', 'q3_time_sec', 'best_quali_time_sec',
       'gap_to_pole', 'made_q2', 'made_q3'],
      dtype='object')

In [106]:
df_qual = df_qual[[ 'qual_ref', 'season', 'round_number', 'race_name', 'circuit_ref', 'race_date',
       'driver_ref', 'constructor_ref', 'quali_position',
       'q1_time_sec', 'q2_time_sec', 'q3_time_sec', 'best_quali_time_sec',
       'gap_to_pole', 'made_q2', 'made_q3']]

In [108]:
df_qual.to_parquet(r"C:\Users\Asus\Desktop\Formula1\data\silver\qualifying\cleaned_qualifying.parquet")